In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.support import expected_conditions as EC
from datetime import datetime, timedelta
import time
from typing import List
from time import sleep
from contextlib import suppress
from io import StringIO
import re


def setup_driver(headless = True):
    options = webdriver.ChromeOptions()
    options.add_argument('--disable-notifications')
    if headless:
        options.add_argument('--headless')
    options.add_argument('--disable-gpu')  # Recommended for headless
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--no-sandbox')  # Bypass OS security model
    options.add_argument('--disable-dev-shm-usage')  # Overcome limited resource problems
    
    # Add a realistic user agent
    options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36')
    
    # Some additional useful options
    options.add_argument('--disable-blink-features=AutomationControlled')  # Hide automation
    options.add_experimental_option('excludeSwitches', ['enable-automation'])  # Hide automation 
    options.add_experimental_option('useAutomationExtension', False)  # Hide automation
    
    driver = webdriver.Chrome(options=options)
    
    # Execute JS to modify navigator.webdriver flag
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    return driver

def get_league_name_and_country(header_text):
    """
    Extracts the league name from header text like 'USA : NCAA Standings' or 'EUROPE : Eurocup Standings'
    Returns just the league name (e.g., 'NCAA' or 'Eurocup')
    """
    try:
        parts = header_text.strip().split(':')[0]
        parts = parts.replace("\n", ":")
        if len(parts) > 1:
            country = parts.split(":")[1]
            league = parts.split(":")[0]
            return (league, country)
        return (header_text.strip(), "")
    except:
        return (header_text.strip(), "")
    

# def get_exact(league_name, country):
#     country = country.lower()

#     # 🌍 International Competitions
#     if league_name.startswith("FIFA World Cup"): return "WC"
#     if league_name.startswith("World Cup Qualification"): return "WCQ"
#     if league_name.startswith("UEFA Euro"): return "EURO"
#     if league_name.startswith("UEFA Nations League"): return "UNL"
#     if league_name.startswith("Copa America"): return "COPA"
#     if league_name.startswith("Africa Cup of Nations"): return "ACN"
#     if league_name.startswith("Gold Cup"): return "GC"
#     if league_name.startswith("Asian Cup"): return "AC"
#     if league_name.startswith("Olympics Football"): return "OLY"

#     # 🏆 Continental Club Competitions
#     if league_name.startswith("Champions League"): return "CL"
#     if league_name.startswith("Europa League"): return "UEL"
#     if league_name.startswith("Conference League"): return "UECL"
#     if league_name.startswith("UEFA Super Cup"): return "USC"
#     if league_name.startswith("Copa Libertadores"): return "LIB"
#     if league_name.startswith("Copa Sudamericana"): return "SUD"
#     if league_name.startswith("Recopa Sudamericana"): return "REC"
#     if league_name.startswith("CAF Champions League"): return "CAFCL"
#     if league_name.startswith("CAF Confederation Cup"): return "CAFCC"
#     if league_name.startswith("CONCACAF Champions Cup"): return "CCC"
#     if league_name.startswith("AFC Champions League"): return "AFCCL"
#     if league_name.startswith("AFC Cup"): return "AFCC"
#     if league_name.startswith("FIFA Club World Cup"): return "CWC"
#     if league_name.startswith("Club Friendly"): return "CF"

#     # 🇬🇧 England
#     if league_name.startswith("Premier League"): return "EPL"
#     if league_name.startswith("Championship"): return "CH"
#     if league_name.startswith("League One"): return "L1"
#     if league_name.startswith("League Two"): return "L2"
#     if league_name.startswith("FA Cup"): return "FAC"
#     if league_name.startswith("EFL Cup"): return "EFLC"
#     if league_name.startswith("Community Shield"): return "CS"

#     # 🇪🇸 Spain
#     if league_name.startswith("LaLiga"): return "LL"
#     if league_name.startswith("LaLiga2"): return "LL2"
#     if league_name.startswith("Copa del Rey"): return "CDR"
#     if league_name.startswith("Supercopa de España"): return "SCE"

#     # 🇩🇪 Germany
#     if league_name.startswith("Bundesliga"): return "BUN"
#     if league_name.startswith("2. Bundesliga"): return "BUN2"
#     if league_name.startswith("DFB Pokal"): return "DFB"
#     if league_name.startswith("DFL Supercup"): return "DFLS"

#     # 🇮🇹 Italy
#     if league_name.startswith("Serie A"): return "SA"
#     if league_name.startswith("Serie B"): return "SB"
#     if league_name.startswith("Coppa Italia"): return "CI"
#     if league_name.startswith("Supercoppa Italiana"): return "SCI"

#     # 🇫🇷 France
#     if league_name.startswith("Ligue 1"): return "L1"
#     if league_name.startswith("Ligue 2"): return "L2"
#     if league_name.startswith("Coupe de France"): return "CDF"
#     if league_name.startswith("Trophée des Champions"): return "TDC"

#     # 🇵🇹 Portugal
#     if league_name.startswith("Primeira Liga"): return "PRL"
#     if league_name.startswith("Segunda Liga"): return "SEG"
#     if league_name.startswith("Taça de Portugal"): return "TP"
#     if league_name.startswith("Supertaça"): return "ST"

#     # 🇳🇱 Netherlands
#     if league_name.startswith("Eredivisie"): return "ERED"
#     if league_name.startswith("Eerste Divisie"): return "EER"
#     if league_name.startswith("KNVB Beker"): return "KNVB"
#     if league_name.startswith("Johan Cruijff Schaal"): return "JCS"

#     # 🇧🇪 Belgium
#     if league_name.startswith("Jupiler Pro League"): return "JPL"
#     if league_name.startswith("Belgian Cup"): return "BC"
#     if league_name.startswith("Belgian Super Cup"): return "BSC"

#     # 🇷🇺 Russia
#     if league_name.startswith("Russian Premier League"): return "RPL"
#     if league_name.startswith("Russian Cup"): return "RC"
#     if league_name.startswith("Russian Super Cup"): return "RSC"

#     # 🇹🇷 Turkey
#     if league_name.startswith("Super Lig"): return "SL"
#     if league_name.startswith("Turkish Cup"): return "TC"
#     if league_name.startswith("Turkish Super Cup"): return "TSC"

#     # 🇬🇷 Greece
#     if league_name.startswith("Super League Greece"): return "SLG"
#     if league_name.startswith("Greek Cup"): return "GCUP"

#     # 🇺🇸 USA
#     if league_name.startswith("Major League Soccer"): return "MLS"
#     if league_name.startswith("US Open Cup"): return "USOC"
#     if league_name.startswith("MLS Cup"): return "MLSC"

#     # 🇧🇷 Brazil
#     if league_name.startswith("Serie A Betano"): return "SA"
#     if league_name.startswith("Serie B"): return "SB"
#     if league_name.startswith("Copa do Brasil"): return "CDB"
#     if league_name.startswith("Supercopa do Brasil"): return "SCB"

#     # 🇦🇷 Argentina
#     if league_name.startswith("Liga Profesional - Clausura"): return "LPF"
#     if league_name.startswith("Primera Nacional"): return "PN"
#     if league_name.startswith("Copa de la Liga"): return "CDL"
#     if league_name.startswith("Supercopa Argentina"): return "SCA"

#     # 🇲🇽 Mexico
#     if league_name.startswith("Liga MX"): return "LMX"
#     if league_name.startswith("Copa MX"): return "CMX"

#     # 🇨🇳 China
#     if league_name.startswith("Chinese Super League"): return "CSL"
#     if league_name.startswith("Chinese FA Cup"): return "CFA"

#     # 🇯🇵 Japan
#     if league_name.startswith("J1 League"): return "J1"
#     if league_name.startswith("J2 League"): return "J2"
#     if league_name.startswith("Emperor"): return "EC"
#     if league_name.startswith("Japanese Super Cup"): return "JSC"

#     # 🇰🇷 Korea
#     if league_name.startswith("K League 1"): return "KL1"
#     if league_name.startswith("K League 2"): return "KL2"
#     if league_name.startswith("Korean FA Cup"): return "KFAC"

#     # 🇿🇦 South Africa
#     if league_name.startswith("Premier Soccer League"): return "PSL"
#     if league_name.startswith("Nedbank Cup"): return "NBC"

#     # 🏴 Scotland
#     if league_name.startswith("Scottish Premiership"): return "SP"
#     if league_name.startswith("Scottish Cup"): return "SC"
#     if league_name.startswith("Scottish League Cup"): return "SLC"

#     # 🇨🇭 Switzerland
#     if league_name.startswith("Swiss Super League"): return "SSL"

#     # 🇦🇹 Austria
#     if league_name.startswith("Austrian Bundesliga"): return "ABL"

#     # 🇩🇰 Denmark
#     if league_name.startswith("Superliga"): return "DENSL"

#     # 🇸🇪 Sweden
#     if league_name.startswith("Allsvenskan"): return "ALLS"

#     # 🇳🇴 Norway
#     if league_name.startswith("Eliteserien"): return "ELIT"

#     # 🇵🇱 Poland
#     if league_name.startswith("Ekstraklasa"): return "EKS"

#     # 🇨🇿 Czech Republic
#     if league_name.startswith("First League"): return "CFL"

#     # 🇭🇷 Croatia
#     if league_name.startswith("HNL"): return "HNL"

#     # 🇷🇸 Serbia
#     if league_name.startswith("SuperLiga"): return "SSL"

#     # 🇭🇺 Hungary
#     if league_name.startswith("NB I"): return "NBI"

#     # 🇷🇴 Romania
#     if league_name.startswith("Liga I"): return "LI"

#     # 🇧🇬 Bulgaria
#     if league_name.startswith("First League"): return "BFL"

#     # 🇸🇰 Slovakia
#     if league_name.startswith("Super Liga"): return "SVKSL"

#     # 🇸🇮 Slovenia
#     if league_name.startswith("PrvaLiga"): return "PRVA"

#     # 🇺🇦 Ukraine
#     if league_name.startswith("Premier League"): return "UPL"

#     #Canada
#     if league_name.startswith("Canadian Premier League"): return "CPL"

#     return league_name



def is_desired_league(game_element):
    try:
        '''
        Starting from this game element, look backwards through the page until you find the first div that 
        has 'tournament__name' in its class name. Use this information to filter out absent leagues
        '''
        league_header = game_element.find_element(By.XPATH, "./preceding::div[contains(@class, 'headerLeague__wrapper')][1]")
        raw_text = league_header.text.strip()

        league_name, country = get_league_name_and_country(raw_text)

        desired_leagues = [
            # International
            'FIFA World Cup',
            'World Cup Qualification',
            'UEFA Euro',
            'UEFA Euro Qualification',
            'UEFA Nations League',
            'Copa America',
            'Africa Cup of Nations',
            'Africa Cup of Nations Women',
            'AFCON Qualification',
            'Gold Cup',
            'Asian Cup',
            'Asian Cup Qualification',
            'OFC Nations Cup',
            'Confederations Cup',

            # Continental Club Competitions
            'Champions League',
            'Champions League - Qualification',
            'Europa League',
            'Europa League - Qualification',
            'Conference League',
            'Conference League - Qualification',
            'UEFA Super Cup',
            'CAF Champions League',
            'CAF Confederation Cup',
            'CONCACAF Champions Cup',
            'Copa Libertadores',
            'Copa Sudamericana',
            'Recopa Sudamericana',
            'AFC Champions League',
            'AFC Cup',
            'FIFA Club World Cup',
            # 'Club Friendly',

            # Argentina
            'Liga Profesional - Clausura',
            'Primera Nacional',
            'Copa de la Liga',
            'Supercopa Argentina',
            
            #Armenia
            'Premier League',

            #ASIA
            'ASEAN Championship',

            #Australia
            'NPL Northern NSW',
            'NPL Queensland',
            'NPL South Australia',
            'NPL Victoria',
            'NSW League One',
            'Queensland Premier League',
            'Tasmania Northern Championship',
            'Tasmania Southern Championship',
            'Victoria Premier League',
            'Victoria Premier League 2',

            #Austria
            'Bundesliga',
            '2. Liga',
            # 'Regionalliga East',
            # 'Regionalliga West',
            # 'Regionalliga South',
            # 'Regionalliga North',
            # 'Tirol',

            # Belgium
            'Jupiler Pro League',
            'Belgian Cup',
            'Belgian Super Cup',

            #Bolivia
            'Division Profesional',

            # Brazil
            'Serie A Betano',
            'Serie B',
            'Copa Betano do Brasil',
            'Supercopa do Brasil',

            #Bulgaria
            'efbet League',

            #Canada
            'Canadian Premier League',

            #Chile
            'Liga de Primera',

            # China
            'Super League',
            'Chinese FA Cup',
            'League One',
            'League Two',

            #Colombia
            'Primera A - Clausura',

            #Croatia
            'HNL',
            
            #Czech
            'Chance Liga',
            # 'ChNL',

            #Denmark
            'Superliga',
            '1st Division',
            '2nd Division',

            #Ecuador
            'Liga Pro',
            'Serie B',

            # England
            'Premier League',
            'Championship',
            'League One',
            'League Two',
            'FA Cup',
            'EFL Cup',
            'Community Shield',
           
            #Estonia
            'Meistriliiga',
            'Esiliiga',
            # 'Esiliiga B',
            
            #Finland
            'Ykkosliiga',
            'Veikkausliiga',
            'Ykkonen',

            # France
            'Ligue 1',
            'Ligue 2',
            'Coupe de France',
            'Trophée des Champions',

            # Germany
            'Bundesliga',
            '2. Bundesliga',
            'DFB Pokal',
            'DFL Supercup',

            # Greece
            'Super League Greece',
            'Greek Cup',

            #Hungary
            'NB I.',
            'NB II.',

            #Iceland
            'Division 1',
            'Division 2',
            # 'Besta deild Karla',
            'Besta deild Women',

            #Ireland
            'Premier Division',

            #Isreal 
            'Toto Cup',

            # Italy
            'Serie A',
            'Serie B',
            'Coppa Italia',
            'Supercoppa Italiana',
            
            #Indonesia
            # 'President Cup',
            
            # Japan
            'J1 League',
            'J2 League',
            'Emperor\'s Cup',
            'Japanese Super Cup',
            
            #Kazakhstan
            'Premier League',
            'Kazakhstan Cup',

            #Latvia
            'Virsliga',

            # Mexico
            'Liga MX - Apertura',
            'Liga MX - Play Offs',
            'Copa MX',

            # Netherlands
            'Eredivisie',
            'Eerste Divisie',
            'KNVB Beker',
            'Johan Cruijff Schaal',
            
            #Norway
            'Eliteserien',

            #Paraguay
            'Copa de Primera - Clausura',

            #Poland
            'Ekstraklasa',
            'Division 1',
            'Division 2',

            # Portugal
            'Super Cup',
            'Primeira Liga',
            'Segunda Liga',
            'Taça de Portugal',
            'Supertaça Cândido de Oliveira',

            #Romania
            'Superliga',

            # Russia
            'Premier League',
            'FNL',
            'Russian Cup',
            'Russian Super Cup',

            #Scotland
            'Premiership',
            'Championship',
            'League one',
            'League Two',

            #Serbia
            'Mozzart Bet Prva Liga',
            'Mozzart Bet Super Liga',

            #Slovakia
            'Nike liga',
            '2.liga',

            #Slovenia
            'Prva liga',

            # Spain
            'LaLiga',
            'LaLiga2',
            'Copa del Rey',
            'Supercopa de España',

            # 🇿🇦 South Africa
            'Betway Premiership',
            'Nedbank Cup',

            # Korea
            'K League 1',
            'K League 2',
            'Korean Cup',

            #Sweden
            'Allsvenskan',
            'Superettan',

            #Switzerland
            'Super League',

            # Turkey
            'Super Lig',
            'Turkish Cup',
            'Turkish Super Cup',

            #Ukraine
            'Premier League',

            # USA
            'MLS',
            'US Open Cup',
            'MLS Cup',
            'USL Championship',

            #Wales
            'Cymru Premier',
        ]

        # return (any(league_name == league for league in desired_leagues), get_exact(league_name, country), country)   
        return (any(league_name == league for league in desired_leagues), league_name, country)   
    except NoSuchElementException:
        return (False, "", "")
    

def get_upcoming_games(driver, day = 0):
    driver.get("https://www.flashscore.com/football/")

    with suppress(Exception):
        accept_button = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))
        )
        accept_button.click()
    
    upcoming = []

    if day > 0:
        for _ in range(day):
            next = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "button[data-day-picker-arrow='next']"))
            )
            next = driver.find_element(By.CSS_SELECTOR, "button[data-day-picker-arrow='next']")
            driver.execute_script("arguments[0].click();", next)
            sleep(3)

    try:
        # Wait for games to load
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "event__match"))
        )

        games = driver.find_elements(By.CLASS_NAME, "event__match")

        for game in games:
            try:
                is_league, league, country = is_desired_league(game)
                if is_league:
                    home = game.find_element(By.CLASS_NAME, "event__homeParticipant")
                    away = game.find_element(By.CLASS_NAME, "event__awayParticipant")
                    time = game.find_element(By.CLASS_NAME, "event__time")
                    game_link = game.find_element(By.CLASS_NAME, "eventRowLink").get_attribute("href")
                    
                    upcoming.append({
                        'league': league,
                        'country': country,
                        'home': home.text,
                        'away': away.text,
                        'time': time.text[:5],
                        'link': game_link
                    })
            except Exception as e:
                continue 
    except Exception as e:
        print(f"Error getting upcoming games: {e}")
    return upcoming


def get_team_last_matches(driver, element, target_league, section_index, team = "NA"):
    target_league = target_league.lower()
    matches = []

    # Click show more only for the specific section we're currently processing
    length = 1 if section_index < 2 else 0
    for _ in range(length):
        try:            
            show_more_button = element.find_element(By.CLASS_NAME, "wclButtonLink--h2h")
            time.sleep(1)
            driver.execute_script("arguments[0].scrollIntoView(true);", show_more_button)
            driver.execute_script("arguments[0].click();", show_more_button)
            time.sleep(2)
        except Exception as e:
            # print(f'Error clicking show more icon: {e}')
            pass
    

    try:
        # now = datetime.now()
        # cutoff_date = now - timedelta(days = 300)
        
        rows = WebDriverWait(element, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "h2h__row"))
        )
        
        for row in rows:
            try:
                # Extract data needed to determine if we'll process this row
                date_arr = row.find_element(By.CLASS_NAME, "wclH2h__date").text.strip().split()
                date_str = date_arr[0]
                # match_date = datetime.strptime(date_str, '%d.%m.%y')
                league = row.find_element(By.CLASS_NAME, "h2h__event").text.lower()
                
                # Check if we should process this row
                # if match_date < cutoff_date:
                #     break

                # if not target_league.startswith(league):
                #     continue

                # if section_index < 2:
                #     if count > 11:
                #         break

                # else:
                #     if count > 4:
                #         break
                
                # Get basic match data
                match_link = row.get_attribute("href")
                home_team = row.find_element(By.CLASS_NAME, "h2h__homeParticipant").text
                away_team = row.find_element(By.CLASS_NAME, "h2h__awayParticipant").text
                score = row.find_element(By.CLASS_NAME, "h2h__result").text
                score = re.sub(r'\([^)]*\)', "", score).strip().split()

                # Store match data
                match_data = {
                    'link': match_link,
                    'date': date_str,
                    'home': home_team,
                    'away': away_team,
                    'league': league,
                    'home_score': score[0] if len(score) > 0 else '0',
                    'away_score': score[1] if len(score) > 1 else '0',
                }
                matches.append(match_data)
                
            except Exception as e:
                # print(f"Error processing row: {e}")
                continue


    except Exception as e:
        # print(f"Error getting matches: {e}")
        pass
    
    return matches

def get_team_standings(driver, url, home_team, away_team):
    try:
        driver.get(url)
        # Handle cookie consent if present
        with suppress(Exception):
            accept_button = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))
            )
            accept_button.click()
            
        
        sleep(2)
        tab_buttons = driver.find_elements(By.CSS_SELECTOR, "div.detailOver > div > a")

        standings_button = None

        for tab_button in tab_buttons:
            if tab_button.text.startswith("STANDINGS"):
                standings_button = tab_button
                break

        if standings_button is None:
            raise Exception("Standings tab not found!")
        else:
            driver.execute_script("arguments[0].click();", standings_button)

        sleep(2)  # Wait for tab to load
        

        # Get sections with explicit wait
        rows = WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "ui-table__row"))
        )
        

        home_found, away_found = False, False

        #home[0] = standing
        #home[1] = points
        #home[2] = match played
        result = {"home": [None, None, None], "away": [None, None, None]}

        for row in rows:
            team = row.find_element(By.CLASS_NAME, "tableCellParticipant").text
            if team == home_team:
                result["home"][0] = row.find_element(By.CLASS_NAME, "table__cell--rank").text 
                result["home"][1] = row.find_element(By.CLASS_NAME, "table__cell--points").text 
                result["home"][2] = row.find_element(By.CLASS_NAME, "table__cell--value").text 
                home_found = True

            elif team == away_team:
                result["away"][0] = row.find_element(By.CLASS_NAME, "table__cell--rank").text 
                result["away"][1] = row.find_element(By.CLASS_NAME, "table__cell--points").text 
                result["away"][2] = row.find_element(By.CLASS_NAME, "table__cell--value").text
                away_found = True

        if not home_found or not away_found:
            raise Exception("Team not found in standing!")

        return result
        
    except Exception as e:
            print(f"Error in get_team_standings: {e}")
            return {'home': [], 'away': []}

    

def scrape_h2h_page(driver, url, league, home_team, away_team):
    try:
        driver.get(url)
        # Handle cookie consent if present
        with suppress(Exception):
            accept_button = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))
            )
            accept_button.click()
            
        # Click H2H tab and wait for it to load
        sleep(2)
        tab_buttons = driver.find_elements(By.CSS_SELECTOR, "div.detailOver > div > a")

        h2h_button = None

        for tab_button in tab_buttons:
            if tab_button.text.startswith("H2H"):
                h2h_button = tab_button
                break

        if h2h_button is None:
            raise Exception("H2H tab not found!")
        else:
            driver.execute_script("arguments[0].click();", h2h_button)

        sleep(2)  # Wait for tab to load
            

        # Get sections with explicit wait
        sections = WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "h2h__section"))
        )
        if len(sections) < 2:
            raise Exception("Incomplete sections found!\n")
        
        results = {
            'home_matches': get_team_last_matches(driver, sections[0], league, 0, home_team),
            'away_matches': get_team_last_matches(driver, sections[1], league, 1, away_team),
            'h2h_matches': [] if len(sections) < 3 else get_team_last_matches(driver, sections[2], league, 2)
        }

        results['standing'] = get_team_standings(driver, url, home_team, away_team)
        
        return results
        
    except Exception as e:
        # print(f"Error in scrape_h2h_page: {e}")
        return {'home_matches': [], 'away_matches': [], 'h2h_matches': [], 'Standing': {'home': [], 'away': []}}


def main():
    # 0 for today, 1 for next day games
    days = [
        0, 
        # 1, 
        # 2, 
        # 3,
    ]
    
    for day in days:
        driver = setup_driver(True)
        try:
            # Get today's upcoming games
            upcoming = get_upcoming_games(driver, day)

            number_of_games = len(upcoming)

            print(f"upcoming games  = {number_of_games}")

            file = r"C:\Users\HP\source\repos\Rehoboam\Rehoboam\Data\Football\Football.txt"

            # For each upcoming game, get last 15 scores and H2H
            last_saved = 2 #default value should be 0
            
            for number, game in enumerate(upcoming):
                if (number+1) <= last_saved:
                    continue
                print(f'{number+1}/{number_of_games}', '\r', end = '')
                #Filter only alphabets
                home_team = game['home']
                away_team = game['away']
                country   = game['country']
                league    = game['league']
                game_time = game['time']

                results = scrape_h2h_page(driver, game['link'], league, home_team, away_team)

                # print(results)

                output_buffer = StringIO()

                output_buffer.write(f'{home_team} goals scored: ')
                for match in results['home_matches']:
                    if home_team == match['home']:
                        output_buffer.write(match['home_score']+' ')
                    else:
                        output_buffer.write(match['away_score']+' ')
                output_buffer.write('\n')

                output_buffer.write(f'{home_team} goals conceded: ')
                for match in results['home_matches']:
                    if home_team == match['home']:
                        output_buffer.write(match['away_score']+' ')
                    else:
                        output_buffer.write(match['home_score']+' ')
                output_buffer.write('\n')


                output_buffer.write(f'{away_team} goals scored: ')
                for match in results['away_matches']:
                    if away_team == match['home']:
                        output_buffer.write(match['home_score']+' ')
                    else:
                        output_buffer.write(match['away_score']+' ')
                output_buffer.write('\n')

                output_buffer.write(f'{away_team} goals conceded: ')
                for match in results['away_matches']:
                    if away_team == match['home']:
                        output_buffer.write(match['away_score']+' ')
                    else:
                        output_buffer.write(match['home_score']+' ')
                output_buffer.write('\n')

                output_buffer.write(f'H2H {len(results["h2h_matches"])}\n')

                for match in results['h2h_matches']:
                    if home_team == match['home']:
                        home_h2h_score = match['home_score'] + ' 1'
                        away_h2h_score = match['away_score'] + ' 2'
                    else:
                        home_h2h_score = match['away_score'] + ' 2'
                        away_h2h_score = match['home_score'] + ' 1'

                    home_h2h_score += ' ' + match['date']
                    away_h2h_score += ' ' + match['date']

                    output_buffer.write(home_h2h_score+'\n')
                    output_buffer.write(away_h2h_score+'\n')

                if len(results['standing']['home']) > 0 and len(results['standing']['away']) > 0:
                    output_buffer.write(f"Home standing: {results['standing']['home'][0]} \
{results['standing']['home'][1]} \
{results['standing']['home'][2]}\n")
                    output_buffer.write(f"Away standing: {results['standing']['away'][0]} \
{results['standing']['away'][1]} \
{results['standing']['away'][2]}\n")
                else:
                    output_buffer.write(f"Home standing: \n")
                    output_buffer.write(f"Away standing: \n")

                output_buffer.write(f'({country}, {league}, {game_time})\n\n')

                with open(file, 'a') as fileObj:
                    fileObj.write(output_buffer.getvalue())

                
                last_saved = number + 1

        except Exception as e:
            print(f"Error in main: {e}")  
            print(f'Failed trying to get game {number} of {len(upcoming)}')
        finally:
            driver.quit() 
        print(f'Day {day} done')

if __name__ == "__main__":
    main()

    

upcoming games  = 27
Error in get_team_standings: Standings tab not found!


KeyboardInterrupt: 